<!---
SPDX-FileCopyrightText: Copyright 2026, Arm Limited and/or its affiliates.
SPDX-License-Identifier: Apache-2.0
--->

# Ethos-U Python API Walkthrough

This notebook explains the Ethos-U Python API path added in `mlia-ethos-u` and shows how to use it from a local development checkout.

It covers:

- bootstrapping the Ethos-U plugin without installing a wheel
- discovering available Ethos-U targets and backends
- creating a tiny demo model
- using `get_advisor(...)` to obtain an `EthosUInferenceAdvisor`
- using `run_advisor(...)` to get standardized compatibility and performance output in Python


## Scope of this notebook

This Ethos-U notebook is about the target-specific Python API integration for path-based model inputs such as Keras and TensorFlow Lite.

## API layers

There are two useful levels to understand:

1. `get_advisor(...)` gives you the target-specific advisor object. In the Ethos-U case, that is `EthosUInferenceAdvisor`.
2. `run_advisor(...)` is the higher-level core API that executes the workflow and returns standardized JSON-compatible output.

The Ethos-U plugin now also exposes an API event-handler factory so the core API can run Ethos-U analysis in collect-only mode and return structured results directly to Python callers.


## When plugin registration is needed

Manual plugin registration is only needed when you run this notebook directly from a source checkout.

- If `mlia-ethos-u` is installed normally, MLIA should discover the target and backend plugins through package entry points.
- If you run from the repository without installing the package, you need to register the Ethos-U, Vela, and Corstone plugins explicitly so the public API can see them.

This notebook uses the second pattern because it is designed to work from an in-repo development checkout.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint

import tensorflow as tf

try:
    import tf_keras as keras
except ImportError:
    from tensorflow import keras


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists() and current.name == "mlia-ethos-u":
            return current
        current = current.parent
    raise RuntimeError("Run this notebook from inside the mlia-ethos-u repository.")


REPO_ROOT = find_repo_root()
PLUGIN_SRC = REPO_ROOT / "src"
ARTIFACTS_DIR = REPO_ROOT / "notebook_artifacts" / "ethos_u_api_walkthrough"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PLUGIN_SRC) not in sys.path:
    sys.path.insert(0, str(PLUGIN_SRC))

In [ ]:
from mlia.api import (
    get_advisor,
    list_backends,
    list_target_profiles,
    list_targets,
    run_advisor,
    supported_backends,
)
from mlia.backend.corstone.plugin import CorstoneBackendPlugin
from mlia.backend.registry import registry as backend_registry
from mlia.backend.vela.plugin import VelaBackendPlugin
from mlia.core.context import ExecutionContext
from mlia.target.ethos_u.advisor import EthosUInferenceAdvisor
from mlia.target.ethos_u.handlers import EthosUEventHandler
from mlia.target.ethos_u.plugin import (
    EthosUTargetPlugin,
    create_ethos_u_api_event_handler,
)
from mlia.target.registry import registry as target_registry


def register_local_plugins() -> None:
    if "corstone-300" not in backend_registry.items:
        CorstoneBackendPlugin.register(backend_registry)
    if "vela" not in backend_registry.items:
        VelaBackendPlugin.register(backend_registry)
    if "ethos-u55" not in target_registry.items:
        EthosUTargetPlugin.register(target_registry)


register_local_plugins()

print(f"Repository root: {REPO_ROOT}")
print(f"Artifacts dir:    {ARTIFACTS_DIR}")

In [ ]:
ethos_targets = [item for item in list_targets() if "ethos" in json.dumps(item).lower()]

ethos_profiles = {
    target: [
        profile["name"] for profile in profiles if "ethos" in profile["name"].lower()
    ]
    for target, profiles in list_target_profiles().items()
    if "ethos" in target.lower()
}

print("Ethos-U targets discovered through the public MLIA API:")
pprint(ethos_targets)

print("\nEthos-U target profiles:")
pprint(ethos_profiles)

print("\nBackends supported for ethos-u55-256:")
pprint(supported_backends("ethos-u55-256"))

print("\nAll registered backends:")
pprint([backend["name"] for backend in list_backends()])

## Build a tiny demo model

The helper below follows the same shape used in the test fixtures: a tiny convolutional Keras model that we save both as Keras and as TensorFlow Lite.


In [ ]:
def build_demo_model() -> keras.Model:
    model = keras.Sequential(
        [
            keras.Input(shape=(28, 28, 1), batch_size=1, name="input"),
            keras.layers.Reshape((28, 28, 1)),
            keras.layers.Conv2D(12, (3, 3), activation="relu", name="conv1"),
            keras.layers.Conv2D(12, (3, 3), activation="relu", name="conv2"),
            keras.layers.MaxPool2D(2, 2),
            keras.layers.Flatten(),
            keras.layers.Dense(10, name="output"),
        ]
    )
    model.compile(optimizer="sgd", loss="mean_squared_error")
    return model


def convert_to_tflite(model: keras.Model, output_path: Path) -> Path:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    output_path.write_bytes(tflite_model)
    return output_path


demo_model = build_demo_model()
keras_model_path = ARTIFACTS_DIR / "demo_model.h5"
tflite_model_path = ARTIFACTS_DIR / "demo_model.tflite"

demo_model.save(keras_model_path)
convert_to_tflite(demo_model, tflite_model_path)

print("Saved:")
print(f"- {keras_model_path}")
print(f"- {tflite_model_path}")

## The new Ethos-U API hook

The plugin now exposes `create_ethos_u_api_event_handler(...)`. This is the target-side hook that lets the core `run_advisor(...)` path obtain an Ethos-U handler in `collect_only=True` mode.


In [ ]:
api_handler = create_ethos_u_api_event_handler(ARTIFACTS_DIR / "api_handler_output")

print(type(api_handler).__name__)
print(f"collect_only = {api_handler.collect_only}")
print(isinstance(api_handler, EthosUEventHandler))

## `get_advisor(...)`: obtain the Ethos-U advisor directly

This lower-level call is useful when you want to inspect the configured advisor, execution context, or target-specific wiring.


In [ ]:
advisor_context = ExecutionContext(output_dir=ARTIFACTS_DIR / "advisor_output")

advisor = get_advisor(
    advisor_context,
    "ethos-u55-256",
    str(keras_model_path),
    backends=["vela"],
)

print(type(advisor).__name__)
print(isinstance(advisor, EthosUInferenceAdvisor))

print("\nExecutionContext event handlers:")
pprint([type(handler).__name__ for handler in advisor_context.event_handlers or []])

print("\nDirect advisor path uses collect_only=False by default:")
print(advisor_context.event_handlers[0].collect_only)

print("\nConfig parameters prepared by the target plugin:")
pprint(advisor_context.config_parameters)

## `run_advisor(...)`: standardized Python output

The helper below runs the shared MLIA API and catches backend-availability issues so the notebook remains readable even on machines where Vela or Corstone are not installed.

Note that the core `run_advisor(...)` API supports `compatibility` and `performance`. Optimization remains a lower-level target-specific path.


In [ ]:
def try_run_advisor(**kwargs):
    try:
        return run_advisor(**kwargs)
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
        return None


def summarize_result(result: dict | None) -> None:
    if result is None:
        print("No result returned.")
        return

    print(f"schema_version: {result.get('schema_version')}")
    print(f"model keys:      {sorted(result.get('model', {}).keys())}")
    print(f"context keys:    {sorted(result.get('context', {}).keys())}")
    print(
        f"backends:        {[item.get('name') for item in result.get('backends', [])]}"
    )
    print(f"result count:    {len(result.get('results', []))}")


compatibility_result = try_run_advisor(
    advice_category="compatibility",
    target_profile="ethos-u55-256",
    model=str(tflite_model_path),
    backends=["vela"],
)

summarize_result(compatibility_result)

In [ ]:
if compatibility_result is not None:
    print(json.dumps(compatibility_result, indent=2)[:2500])

In [ ]:
performance_result = try_run_advisor(
    advice_category="performance",
    target_profile="ethos-u55-256",
    model=str(tflite_model_path),
    backends=["vela"],
)

summarize_result(performance_result)

In [ ]:
if performance_result is not None:
    print(json.dumps(performance_result, indent=2)[:2500])

## Optional: compare backend selections

If your environment includes Corstone as well as Vela, you can request both and inspect the merged standardized output.


In [ ]:
combined_result = try_run_advisor(
    advice_category="performance",
    target_profile="ethos-u55-256",
    model=str(tflite_model_path),
    backends=["vela", "corstone-310"],
)

summarize_result(combined_result)

## Takeaways

- `get_advisor(...)` resolves Ethos-U correctly and returns `EthosUInferenceAdvisor`.
- the Ethos-U plugin now provides an API event-handler factory through `create_ethos_u_api_event_handler(...)`.
- `run_advisor(...)` can use that target-side hook to return structured Ethos-U compatibility or performance output directly to Python.
- for optimization workflows or deeper target-specific control, stay on the lower-level advisor path.
